In [1]:
"""
OMDb API Client - Jupyter Notebook версия

Улучшенный скрипт для поиска фильмов через OMDb API
с полной обработкой ошибок и валидацией данных.
"""

import os
import json
import logging
import requests
from dotenv import load_dotenv

# ==================== КОНФИГУРАЦИЯ ====================

# API параметры
OMDB_API_URL = "https://www.omdbapi.com/"  # ✅ HTTPS вместо HTTP
REQUEST_TIMEOUT = 10

# Поля результатов
MOVIE_FIELDS = {
    "Title": "Название",
    "Year": "Год",
    "Genre": "Жанр",
    "Director": "Режиссёр",
    "imdbRating": "Рейтинг IMDb",
    "Actors": "Актёры",
}

# Логирование для Jupyter
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)
logger = logging.getLogger(__name__)


/Users/egorlubar/PyCharmMiscProject/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# ==================== ОСНОВНЫЕ ФУНКЦИИ ====================


def validate_title(title: str) -> str:
    """
    Проверка и очистка названия фильма.

    Args:
        title: Введённое пользователем название

    Returns:
        Очищенное название

    Raises:
        ValueError: Если название некорректно
    """
    if not isinstance(title, str):
        raise ValueError("Название должно быть строкой")

    title = title.strip()

    if not title:
        raise ValueError("Название фильма не может быть пустым")

    if len(title) > 100:
        raise ValueError("Название слишком длинное (максимум 100 символов)")

    logger.info(f"✓ Название валидно: '{title}'")
    return title


def load_api_key() -> str:
    """
    Загрузка и проверка API-ключа из .env файла.

    Returns:
        API-ключ OMDb

    Raises:
        ValueError: Если ключ не найден или некорректен
    """
    load_dotenv()

    api_key = os.getenv("OMDB_API_KEY")

    if not api_key:
        raise ValueError(
            "❌ OMDB_API_KEY не найден в .env файле.\n"
            "Пожалуйста, создайте .env и добавьте: OMDB_API_KEY=your_key"
        )

    if len(api_key.strip()) < 5:
        raise ValueError("❌ API-ключ выглядит некорректно")

    logger.info("✓ API-ключ загружен")
    return api_key


def fetch_movie(title: str, api_key: str) -> dict:
    """
    Запрос к OMDb API с полной обработкой ошибок.

    Args:
        title: Название фильма для поиска
        api_key: API-ключ OMDb

    Returns:
        Словарь с данными о фильме

    Raises:
        TimeoutError: Если превышено время ожидания
        ConnectionError: Если ошибка подключения
        RuntimeError: При других ошибках запроса
        ValueError: Если ошибка разбора JSON или фильм не найден
    """
    logger.info(f"🔍 Поиск фильма: '{title}'...")

    try:
        response = requests.get(
            OMDB_API_URL,
            params={"t": title, "apikey": api_key},
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()

    except requests.exceptions.Timeout:
        raise TimeoutError(f"Превышено время ожидания ({REQUEST_TIMEOUT} сек)")

    except requests.exceptions.ConnectionError as e:
        raise ConnectionError(f"Ошибка подключения к OMDb: {e}")

    except requests.exceptions.HTTPError:
        raise RuntimeError(f"Ошибка HTTP при обращении к OMDb")

    except requests.exceptions.RequestException as e:
        raise RuntimeError(f"Ошибка запроса: {e}")

    # Парсинг JSON ответа
    try:
        data = response.json()
    except json.JSONDecodeError as e:
        raise ValueError(f"Ошибка парсинга JSON ответа: {e}")

    # Проверка ответа API
    if data.get("Response") == "False":
        error_msg = data.get("Error", "Фильм не найден")
        raise ValueError(f"❌ {error_msg}")

    logger.info("✓ Фильм найден успешно")
    return data


def display_movie_info(movie_data: dict) -> None:
    """
    Красивый форматированный вывод информации о фильме.

    Args:
        movie_data: Словарь с данными о фильме
    """
    print("\n" + "=" * 70)
    print("📽️  ИНФОРМАЦИЯ О ФИЛЬМЕ")
    print("=" * 70)

    for api_field, display_name in MOVIE_FIELDS.items():
        value = movie_data.get(api_field)

        # Обработка отсутствующих значений
        if value is None or value == "N/A":
            value = "⚠️  Информация недоступна"

        print(f"  {display_name:.<25} {value}")

    print("=" * 70 + "\n")


def print_summary(movie_data: dict) -> None:
    """
    Краткое резюме о фильме.

    Args:
        movie_data: Словарь с данными о фильме
    """
    title = movie_data.get("Title", "Unknown")
    year = movie_data.get("Year", "N/A")
    genre = movie_data.get("Genre", "N/A")
    director = movie_data.get("Director", "N/A")

    summary = f"✓ '{title}' ({year}) — {genre}. Режиссёр: {director}."
    print(f"📌 {summary}\n")



In [3]:
# ==================== ГЛАВНЫЙ КОД ====================

def search_and_display():
    """Интерактивный поиск и отображение информации о фильме."""
    try:
        # Загрузка конфигурации
        print("⚙️  Инициализация...\n")
        api_key = load_api_key()

        # Ввод от пользователя с валидацией
        user_input = input("🎬 Введите название фильма: ")
        title = validate_title(user_input)

        # Поиск фильма
        movie = fetch_movie(title, api_key)

        # Вывод результатов
        display_movie_info(movie)
        print_summary(movie)

        print("✅ Поиск завершён успешно!\n")
        return movie

    except ValueError as e:
        print(f"\n❌ Ошибка валидации: {e}\n")
        return None

    except (TimeoutError, ConnectionError) as e:
        print(f"\n❌ Ошибка сети: {e}\n")
        return None

    except RuntimeError as e:
        print(f"\n❌ Ошибка: {e}\n")
        return None

    except KeyboardInterrupt:
        print("\n\n⏹️  Операция прервана пользователем\n")
        return None

    except Exception as e:
        print(f"\n❌ Непредвиденная ошибка: {e}\n")
        logger.exception("Детали ошибки:")
        return None


# Запуск поиска
result = search_and_display()


INFO: ✓ API-ключ загружен


⚙️  Инициализация...



INFO: ✓ Название валидно: 'Spider Man'
INFO: 🔍 Поиск фильма: 'Spider Man'...
INFO: ✓ Фильм найден успешно



📽️  ИНФОРМАЦИЯ О ФИЛЬМЕ
  Название................. Spider Man: Lost Cause
  Год...................... 2014
  Жанр..................... Action, Adventure, Comedy
  Режиссёр................. Joey Lever
  Рейтинг IMDb............. 4.3
  Актёры................... Joey Lever, Craig Ellis, Teravis Ward

📌 ✓ 'Spider Man: Lost Cause' (2014) — Action, Adventure, Comedy. Режиссёр: Joey Lever.

✅ Поиск завершён успешно!

